In [ ]:
!pip install transformers datasets torch scikit-learn gradio seaborn matplotlib accelerate -q
print("All libraries installed successfully.")

In [ ]:
# Standard Python libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# HuggingFace — dataset loading
from datasets import load_dataset

# HuggingFace — transformer model and tokenizer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# PyTorch — deep learning backend
import torch
from torch.utils.data import Dataset

# Scikit-learn — baseline model and evaluation
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)
from sklearn.preprocessing import LabelEncoder

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in use: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
print("Loading GoEmotions dataset from HuggingFace...")
dataset = load_dataset("go_emotions", "simplified")

print(dataset)

print("\nSample entry from training set:")
sample = dataset['train'][0]
print(f"  Text   : {sample['text']}")
print(f"  Label  : {sample['labels']}")
print(f"  ID     : {sample['id']}")

print(f"\nTraining examples  : {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"Test examples      : {len(dataset['test'])}")

In [ ]:
KEEP_EMOTIONS = [
    'joy', 'sadness', 'anger', 'fear', 'disgust',
    'surprise', 'gratitude', 'love', 'curiosity',
    'confusion', 'disappointment', 'annoyance', 'neutral'
]

# Original 28 emotion labels
all_emotions = [
    'admiration','amusement','anger','annoyance','approval','caring',
    'confusion','curiosity','desire','disappointment','disapproval',
    'disgust','embarrassment','excitement','fear','gratitude','grief',
    'joy','love','nervousness','optimism','pride','realization',
    'relief','remorse','sadness','surprise','neutral'
]

# Build mapping: old index → new index (or None if dropping)
keep_indices = [all_emotions.index(e) for e in KEEP_EMOTIONS]
old_to_new = {}
for new_idx, old_idx in enumerate(keep_indices):
    old_to_new[old_idx] = new_idx

print("Keeping emotions:")
for i, e in enumerate(KEEP_EMOTIONS):
    print(f"  [{i}] {e}")

def filter_and_remap(example):
    label = example['labels']
    if isinstance(label, list):
        label = label[0] if len(label) > 0 else all_emotions.index('neutral')
    if isinstance(label, torch.Tensor):
        label = label.item()
    return label in keep_indices

def remap_label(example):
    label = example['labels']
    if isinstance(label, list):
        label = label[0]
    if isinstance(label, torch.Tensor):
        label = label.item()
    return {'labels': old_to_new[int(label)]}

print("\nFiltering dataset...")
dataset = dataset.filter(filter_and_remap)
dataset = dataset.map(remap_label)

# Update global variables
emotion_labels = KEEP_EMOTIONS
NUM_LABELS = len(KEEP_EMOTIONS)

print(f"\nDataset size after filtering:")
print(f"  Train : {len(dataset['train'])}")
print(f"  Val   : {len(dataset['validation'])}")
print(f"  Test  : {len(dataset['test'])}")
print(f"\nClasses reduced: 28 → {NUM_LABELS}")

In [ ]:
emotion_labels = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
NUM_LABELS = len(emotion_labels)
print(f"Total emotion classes: {NUM_LABELS}")
print(f"Emotions: {emotion_labels}")

# Convert dataset to pandas for easy exploration
def extract_first_label(examples):
    result = []
    for l in examples['labels']:
        if isinstance(l, list):
            result.append(l[0] if len(l) > 0 else 27)
        else:
            result.append(int(l))
    return {'label': result}

dataset = dataset.map(extract_first_label, batched=True)

# Build a DataFrame from training data for analysis
train_df = pd.DataFrame({
    'text': dataset['train']['text'],
    'label': dataset['train']['label']
})
train_df['emotion'] = train_df['label'].apply(lambda x: emotion_labels[x])
train_df['text_length'] = train_df['text'].apply(len)

print(f"\nAverage text length : {train_df['text_length'].mean():.0f} characters")
print(f"Max text length     : {train_df['text_length'].max()} characters")
print(f"Min text length     : {train_df['text_length'].min()} characters")

# Plot emotion distribution
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart: count per emotion
emotion_counts = train_df['emotion'].value_counts()
axes[0].bar(range(len(emotion_counts)), emotion_counts.values, color='steelblue', alpha=0.8)
axes[0].set_xticks(range(len(emotion_counts)))
axes[0].set_xticklabels(emotion_counts.index, rotation=90, fontsize=9)
axes[0].set_title('Training set: samples per emotion class', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].axhline(y=emotion_counts.mean(), color='red', linestyle='--', label=f'Mean: {emotion_counts.mean():.0f}')
axes[0].legend()

# Histogram: text length distribution
axes[1].hist(train_df['text_length'], bins=50, color='coral', alpha=0.8)
axes[1].set_title('Distribution of text lengths', fontsize=13)
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Count')
axes[1].axvline(x=train_df['text_length'].mean(), color='navy', linestyle='--',
                label=f"Mean: {train_df['text_length'].mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig('data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nNote the class imbalance — 'neutral' dominates. This is why we use F1 score.")

In [ ]:
print("=" * 60)
print("BASELINE MODEL: TF-IDF + Logistic Regression")
print("=" * 60)

X_train = dataset['train']['text']
y_train = dataset['train']['label']

X_val = dataset['validation']['text']
y_val = dataset['validation']['label']

X_test = dataset['test']['text']
y_test = dataset['test']['label']

print(f"Training samples  : {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples      : {len(X_test)}")

# max_features=50000: keep only the 50,000 most common words
# ngram_range=(1,2): use single words AND two-word phrases (bigrams)
# sublinear_tf=True: apply log normalization to term frequency
print("\nFitting TF-IDF vectorizer...")
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{2,}',  # ignore single characters
    min_df=2  # ignore words appearing in fewer than 2 documents
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(f"Vocabulary size: {len(tfidf.vocabulary_)} unique tokens")
print(f"Training matrix shape: {X_train_tfidf.shape}")
print(f"  (rows=samples, cols=vocabulary size)")

# C=5: regularisation strength (higher C = less regularisation)
# max_iter=1000: max iterations for solver to converge
# multi_class='ovr': one-vs-rest strategy for 28 classes
print("\nTraining Logistic Regression classifier...")
print("(This takes about 30-60 seconds)")
lr_model = LogisticRegression(
    C=5,
    max_iter=1000,
    multi_class='ovr',
    solver='saga',
    n_jobs=-1,  # use all CPU cores
    random_state=42
)
lr_model.fit(X_train_tfidf, y_train)
print("Training complete.")

y_val_pred = lr_model.predict(X_val_tfidf)

baseline_acc = accuracy_score(y_val, y_val_pred)
baseline_f1_macro = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
baseline_f1_weighted = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)

print("\n" + "=" * 40)
print("BASELINE RESULTS (Validation Set)")
print("=" * 40)
print(f"Accuracy         : {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")
print(f"Macro F1         : {baseline_f1_macro:.4f} ({baseline_f1_macro*100:.1f}%)")
print(f"Weighted F1      : {baseline_f1_weighted:.4f} ({baseline_f1_weighted*100:.1f}%)")
print("\nNote: Macro F1 is our primary metric (treats all 28 classes equally)")
print("These numbers will go up significantly after DistilBERT fine-tuning.")

In [ ]:
print("Loading DistilBERT tokenizer...")
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128  # max tokens per sequence (most Reddit comments are shorter)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Show what tokenization looks like on an example
example_text = "I can't believe I finally got the job after 6 months!"
tokens = tokenizer.tokenize(example_text)
token_ids = tokenizer.encode(example_text)
print(f"\nExample text: '{example_text}'")
print(f"Tokens      : {tokens}")
print(f"Token IDs   : {token_ids}")
print(f"Note: 101=[CLS], 102=[SEP], ## prefix means subword continuation")

# Define tokenization function to apply to the whole dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',   # pad all sequences to MAX_LENGTH
        truncation=True,        # cut sequences longer than MAX_LENGTH
        max_length=MAX_LENGTH,
        return_tensors=None     # return Python lists, not tensors (HuggingFace handles conversion)
    )

# Apply tokenization to all splits
# batched=True processes 1000 examples at a time — much faster
print("\nTokenizing training set...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets = tokenized_datasets.remove_columns(['label'])

# Set the format to PyTorch tensors for training
tokenized_datasets.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

print("Tokenization complete.")
print(f"\nTraining set columns : {tokenized_datasets['train'].column_names}")
print(f"Sample input_ids     : {tokenized_datasets['train'][0]['input_ids'][:15]}...")
print(f"Sample attention_mask: {tokenized_datasets['train'][0]['attention_mask'][:15]}...")
print(f"Sample label         : {tokenized_datasets['train'][0]['labels'].item()} = {emotion_labels[tokenized_datasets['train'][0]['labels'].item()]}")

In [ ]:
print("Loading DistilBERT with classification head...")
print(f"Model: {MODEL_NAME}")
print(f"Number of output classes: {NUM_LABELS}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: label for i, label in enumerate(emotion_labels)},
    label2id={label: i for i, label in enumerate(emotion_labels)}
)

# Move model to GPU
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel architecture summary:")
print(f"  Total parameters    : {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Device              : {next(model.parameters()).device}")

# Show model structure
print(f"\nModel layers:")
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {params:,} parameters")

print("\nReady for fine-tuning.")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    weighted_f1 = f1_score(labels, predictions, average='weighted', zero_division=0)

    return {
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    }

print("compute_metrics function defined.")
print("This will be called after each epoch during training.")

# Quick sanity check
fake_logits = np.random.randn(10, 28)  # 10 examples, 28 classes
fake_labels = np.random.randint(0, 28, size=10)
test_result = compute_metrics((fake_logits, fake_labels))
print(f"\nSanity check with random data: {test_result}")
print("(These numbers are random — just verifying the function runs correctly)")

In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='./emotion_model_checkpoints',

    learning_rate=2e-5,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,  # can be larger for eval (no gradient storage)

    num_train_epochs=3,

    weight_decay=0.01,

    warmup_ratio=0.1,

    eval_strategy='epoch',

    save_strategy='epoch',

    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,

    logging_dir='./logs',
    logging_steps=100,

    fp16=True if device.type == 'cuda' else False,

    seed=42,

    report_to='none'
)

# Calculate training steps for reference
steps_per_epoch = len(tokenized_datasets['train']) // 32
total_steps = steps_per_epoch * 3
warmup_steps = int(total_steps * 0.1)

print("Training configuration:")
print(f"  Learning rate      : {training_args.learning_rate}")
print(f"  Batch size (train) : {training_args.per_device_train_batch_size}")
print(f"  Epochs             : {training_args.num_train_epochs}")
print(f"  Weight decay       : {training_args.weight_decay}")
print(f"  Mixed precision fp16: {training_args.fp16}")
print(f"\nEstimated steps:")
print(f"  Steps per epoch    : ~{steps_per_epoch}")
print(f"  Total steps        : ~{total_steps}")
print(f"  Warmup steps       : ~{warmup_steps}")
print(f"\nEstimated training time: 45-60 minutes on T4 GPU")

In [ ]:
print("=" * 60)
print("STARTING FINE-TUNING")
print("=" * 60)
print("This will take approximately 45-60 minutes on T4 GPU.")
print("You will see a progress table appear and update after each epoch.")

# Fix labels to ensure all are single integers not lists
def fix_labels(example):
    label = example['labels']
    if isinstance(label, torch.Tensor):
        label = label.flatten()[0].item()
    elif isinstance(label, list):
        label = label[0] if len(label) > 0 else 27
    return {'labels': int(label)}

tokenized_datasets = tokenized_datasets.map(fix_labels)
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Start training
train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Total training time : {train_result.metrics['train_runtime']:.0f} seconds ({train_result.metrics['train_runtime']/60:.1f} minutes)")
print(f"Samples per second  : {train_result.metrics['train_samples_per_second']:.1f}")
print(f"Total steps trained : {train_result.metrics['train_steps_per_second'] * train_result.metrics['train_runtime']:.0f}")

In [ ]:
SAVE_PATH = './emotion_model_final'

print(f"Saving model to: {SAVE_PATH}")
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

import os
saved_files = os.listdir(SAVE_PATH)
print(f"\nSaved files:")
for f in saved_files:
    size = os.path.getsize(f'{SAVE_PATH}/{f}') / 1e6
    print(f"  {f}: {size:.1f} MB")

print("\nModel saved. You can reload it later with:")
print(f"  model = AutoModelForSequenceClassification.from_pretrained('{SAVE_PATH}')")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{SAVE_PATH}')")

In [ ]:
print("=" * 60)
print("FINAL EVALUATION ON TEST SET")
print("=" * 60)

print("\nRunning DistilBERT on test set...")
test_results = trainer.predict(tokenized_datasets['test'])

# Get predictions
bert_logits = test_results.predictions
bert_preds = np.argmax(bert_logits, axis=-1)
true_labels = test_results.label_ids

# Compute final metrics
bert_acc = accuracy_score(true_labels, bert_preds)
bert_macro_f1 = f1_score(true_labels, bert_preds, average='macro', zero_division=0)
bert_weighted_f1 = f1_score(true_labels, bert_preds, average='weighted', zero_division=0)

print("Running Baseline on test set...")
X_test_tfidf_eval = tfidf.transform(dataset['test']['text'])
baseline_test_preds = lr_model.predict(X_test_tfidf_eval)

baseline_test_acc = accuracy_score(true_labels, baseline_test_preds)
baseline_test_macro_f1 = f1_score(true_labels, baseline_test_preds, average='macro', zero_division=0)
baseline_test_weighted_f1 = f1_score(true_labels, baseline_test_preds, average='weighted', zero_division=0)

print("\n" + "=" * 55)
print(f"{'Metric':<25} {'Baseline':>12} {'DistilBERT':>12}")
print("-" * 55)
print(f"{'Accuracy':<25} {baseline_test_acc*100:>11.1f}% {bert_acc*100:>11.1f}%")
print(f"{'Macro F1 (PRIMARY)':<25} {baseline_test_macro_f1*100:>11.1f}% {bert_macro_f1*100:>11.1f}%")
print(f"{'Weighted F1':<25} {baseline_test_weighted_f1*100:>11.1f}% {bert_weighted_f1*100:>11.1f}%")
print("=" * 55)
improvement = (bert_macro_f1 - baseline_test_macro_f1) * 100
print(f"\nMacro F1 improvement: +{improvement:.1f} percentage points")
print("This improvement = value added by transfer learning (pretrained weights)")

per_class_f1 = f1_score(true_labels, bert_preds, average=None, zero_division=0)
per_class_df = pd.DataFrame({
    'emotion': emotion_labels,
    'f1_score': per_class_f1,
    'count': [sum(np.array(true_labels) == i) for i in range(NUM_LABELS)]
}).sort_values('f1_score', ascending=False)

print("\n--- Per-class F1 scores (DistilBERT) ---")
print(f"{'Emotion':<20} {'F1 Score':>10} {'Test Count':>12}")
print("-" * 45)
for _, row in per_class_df.iterrows():
    bar = '█' * int(row['f1_score'] * 20)
    print(f"{row['emotion']:<20} {row['f1_score']:>8.3f}   {int(row['count']):>8}  {bar}")

In [ ]:
history = trainer.state.log_history

train_logs = [h for h in history if 'loss' in h and 'eval_loss' not in h]
eval_logs  = [h for h in history if 'eval_loss' in h]

# Extract values
train_steps  = [h['step'] for h in train_logs]
train_losses = [h['loss'] for h in train_logs]

eval_epochs   = [h['epoch'] for h in eval_logs]
eval_losses   = [h['eval_loss'] for h in eval_logs]
eval_macro_f1 = [h.get('eval_macro_f1', 0) for h in eval_logs]
eval_accuracy = [h.get('eval_accuracy', 0) for h in eval_logs]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Training loss over steps
axes[0].plot(train_steps, train_losses, color='steelblue', linewidth=1.5, alpha=0.8)
axes[0].set_title('Training Loss over Steps', fontsize=13)
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(bottom=0)

# Plot 2: Validation loss per epoch
axes[1].plot(eval_epochs, eval_losses, 'o-', color='coral', linewidth=2, markersize=8)
axes[1].set_title('Validation Loss per Epoch', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(eval_epochs)

# Plot 3: Validation Macro F1 per epoch
axes[2].plot(eval_epochs, [f*100 for f in eval_macro_f1], 's-', color='green',
             linewidth=2, markersize=8, label='DistilBERT')
axes[2].axhline(y=baseline_test_macro_f1*100, color='red', linestyle='--',
                linewidth=1.5, label=f'Baseline ({baseline_test_macro_f1*100:.1f}%)')
axes[2].set_title('Validation Macro-F1 per Epoch', fontsize=13)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro F1 (%)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_xticks(eval_epochs)

plt.suptitle('Emotion Detection — Training Progress', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved to training_curves.png")

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(true_labels, bert_preds)

# Normalise by row (true labels) so each row sums to 1
cm_normalized = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# Plot 1: Normalised confusion matrix
sns.heatmap(
    cm_normalized,
    annot=False,
    fmt='.2f',
    cmap='Blues',
    xticklabels=emotion_labels,
    yticklabels=emotion_labels,
    ax=axes[0],
    linewidths=0.3,
    linecolor='white'
)
axes[0].set_title('Confusion Matrix (Normalised)\nDistilBERT Emotion Classifier', fontsize=13)
axes[0].set_ylabel('True Emotion', fontsize=11)
axes[0].set_xlabel('Predicted Emotion', fontsize=11)
axes[0].tick_params(axis='x', rotation=90, labelsize=8)
axes[0].tick_params(axis='y', rotation=0, labelsize=8)

# Plot 2: Per-class F1 comparison (Baseline vs DistilBERT)
baseline_per_class_f1 = f1_score(true_labels, baseline_test_preds, average=None, zero_division=0)
x = np.arange(NUM_LABELS)
width = 0.35
axes[1].bar(x - width/2, baseline_per_class_f1, width, label='TF-IDF + LR (Baseline)',
            color='coral', alpha=0.8)
axes[1].bar(x + width/2, per_class_f1, width, label='DistilBERT (Fine-tuned)',
            color='steelblue', alpha=0.8)
axes[1].set_xlabel('Emotion Class', fontsize=11)
axes[1].set_ylabel('F1 Score', fontsize=11)
axes[1].set_title('Per-class F1: Baseline vs DistilBERT', fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(emotion_labels, rotation=90, fontsize=8)
axes[1].legend(fontsize=11)
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved to confusion_matrix.png")
print("\nObservations to mention in your exam:")
print("  - Bright diagonal = correct predictions")
print("  - grief/sadness/disappointment cluster = emotionally similar")
print("  - excitement/joy/amusement cluster = positive emotions confused")
print("  - DistilBERT beats baseline on almost every individual emotion class")

In [ ]:
print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

test_texts = dataset['test']['text']

errors = []
for i, (true, pred) in enumerate(zip(true_labels, bert_preds)):
    if true != pred:
        probs = torch.softmax(torch.tensor(bert_logits[i]), dim=-1)
        confidence = probs[pred].item()
        errors.append({
            'text': test_texts[i],
            'true_emotion': emotion_labels[true],
            'pred_emotion': emotion_labels[pred],
            'confidence': confidence
        })

error_df = pd.DataFrame(errors)
total_errors = len(error_df)
total_test = len(true_labels)
print(f"Total test examples  : {total_test}")
print(f"Correct predictions  : {total_test - total_errors} ({(total_test-total_errors)/total_test*100:.1f}%)")
print(f"Wrong predictions    : {total_errors} ({total_errors/total_test*100:.1f}%)")

# Most common error patterns
print("\n--- Top 10 most common confusion pairs ---")
confusion_pairs = error_df.groupby(['true_emotion', 'pred_emotion']).size().reset_index(name='count')
confusion_pairs = confusion_pairs.sort_values('count', ascending=False).head(10)
print(f"{'True Emotion':<20} {'Predicted As':<20} {'Count':>8}")
print("-" * 50)
for _, row in confusion_pairs.iterrows():
    print(f"{row['true_emotion']:<20} {row['pred_emotion']:<20} {row['count']:>8}")

# Show 5 interesting error examples
print("\n--- Example wrong predictions (high confidence errors) ---")
high_conf_errors = error_df.sort_values('confidence', ascending=False).head(10)
for i, (_, row) in enumerate(high_conf_errors.iterrows()):
    if i >= 5:
        break
    print(f"\nExample {i+1}:")
    print(f"  Text          : '{row['text'][:100]}...' " if len(row['text']) > 100 else f"  Text          : '{row['text']}'")
    print(f"  True emotion  : {row['true_emotion']}")
    print(f"  Predicted as  : {row['pred_emotion']} (confidence: {row['confidence']:.2%})")

print("\n--- Emotions with lowest F1 (hardest to predict) ---")
worst_emotions = per_class_df.tail(5)
for _, row in worst_emotions.iterrows():
    print(f"  {row['emotion']:<20} F1={row['f1_score']:.3f}  (only {int(row['count'])} test examples)")
print("\nNote: Low F1 on rare emotions is expected — few training examples = hard to learn")

In [ ]:
print("\n" + "=" * 65)
print("FINAL RESULTS SUMMARY")
print("Multi-Class Emotion Detection — GoEmotions (28 classes)")
print("=" * 65)
print(f"{'':5} {'Model':<30} {'Accuracy':>10} {'Macro F1':>10} {'Weighted F1':>12}")
print("-" * 65)
print(f"{'[1]':<5} {'TF-IDF + Logistic Regression':<30} {baseline_test_acc*100:>9.1f}% {baseline_test_macro_f1*100:>9.1f}% {baseline_test_weighted_f1*100:>11.1f}%")
print(f"{'[2]':<5} {'DistilBERT (fine-tuned, 3 epochs)':<30} {bert_acc*100:>9.1f}% {bert_macro_f1*100:>9.1f}% {bert_weighted_f1*100:>11.1f}%")
print("-" * 65)
print(f"{'':5} {'Improvement ([2] vs [1])':<30} {(bert_acc-baseline_test_acc)*100:>+9.1f}% {(bert_macro_f1-baseline_test_macro_f1)*100:>+9.1f}%  {(bert_weighted_f1-baseline_test_weighted_f1)*100:>+10.1f}%")
print("=" * 65)
print(f"\nDataset   : GoEmotions (Google, 2020)")
print(f"Classes   : 28 emotion categories")
print(f"Train size: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")
print(f"Model     : distilbert-base-uncased (66M params, 6 transformer layers)")
print(f"Epochs    : 3 | LR: 2e-5 | Batch: 32 | Warmup: 10%")
print(f"Metric    : Macro F1 (primary) — treats all 28 classes equally")
print("\nKey findings:")
print(f"  1. DistilBERT achieves +{(bert_macro_f1-baseline_test_macro_f1)*100:.1f}pp improvement in Macro F1 over the baseline")
print(f"  2. Best predicted emotion: {per_class_df.iloc[0]['emotion']} (F1={per_class_df.iloc[0]['f1_score']:.3f})")
print(f"  3. Hardest emotion: {per_class_df.iloc[-1]['emotion']} (F1={per_class_df.iloc[-1]['f1_score']:.3f}) — fewest training examples")
print(f"  4. Most common error: {confusion_pairs.iloc[0]['true_emotion']} confused with {confusion_pairs.iloc[0]['pred_emotion']}")
print(f"     (emotionally similar classes — expected and explainable)")

In [ ]:
# Load saved model for inference
inference_model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
inference_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
inference_model = inference_model.to(device)
inference_model.eval()

import gradio as gr
import torch
import numpy as np

def predict_emotion_debiased(text: str):
    if not text.strip():
        return {emotion: 0.0 for emotion in emotion_labels}

    inputs = inference_tokenizer(
        text,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = inference_model(**inputs)

    logits = outputs.logits.squeeze()

    # Find neutral index and penalise it
    neutral_idx = emotion_labels.index('neutral')
    logits[neutral_idx] -= 3.0  # push neutral score down

    probs = torch.softmax(logits, dim=-1).cpu().numpy()

    return {emotion_labels[i]: float(probs[i]) for i in range(NUM_LABELS)}

# Test it on a few sentences
test_sentences = [
    "I finally finished my thesis and I can't stop smiling!",
    "I packed up his room today. It still smells like him.",
    "He lied to my face and expects me to just move on.",
    "I heard footsteps behind me and my whole body went cold.",
    "You stayed up all night helping me and I will never forget that."
]

print("Testing debiased predictions:\n")
for sentence in test_sentences:
    result = predict_emotion_debiased(sentence)
    top3 = sorted(result.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"Text: '{sentence[:60]}...'")
    for emotion, prob in top3:
        bar = '█' * int(prob * 25)
        print(f"  {emotion:<15} {prob:.2%}  {bar}")
    print()